# Making list of filenames

In [1]:
import matplotlib.pyplot as plt
from obspy import read_events
from obspy import UTCDateTime
import numpy as np
import pandas as pd
import scipy.stats as stats
import wget
import csv

In [2]:
# Catalog Data
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
filenames = []

j=1
for event in big10_catalog: # For each earthquake

    # Read earthquake data
    origin = event.preferred_origin() or event.origins[0]
    event_time = origin.time
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    moment_tensor = focal_mech.moment_tensor
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1
    plane2 = nodal_planes.nodal_plane_2
    
    mag = event.preferred_magnitude() or event.magnitudes[0]
    mag = float(mag.mag)

    for plane in (plane1,plane2):
        filenames.append(f"Greece_EQ{j}_M{mag}_{plane}_{event_time}.csv")
    j=j+1

    df = pd.DataFrame(filenames)
    df.to_csv("filenames.csv", index=False, header=None)

# Downloading the files

In [3]:
# Catalog Data
filenames = pd.read_csv("filenames.csv", names=['col'], header=None)
filenames=filenames['col'].tolist()
print(filenames)

['Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=120.0, dip=86.0, rake=87.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=339.0, dip=3.0, rake=130.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=119.0, dip=88.0, rake=88.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=73.0, dip=85.0, rake=-177.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=343.0, dip=87.0, rake=-5.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ6_M6.5_N

In [4]:
EQ=1
for i in range(20):
    filename=filenames[i]
    print(EQ)
    print(f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    print(filename)
    relevant_stations = pd.read_csv(f"Finite/{filename}")
    sta_ids = relevant_stations["Station_ID"]
    for station in sta_ids:
        url = f"https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/EU/{station.upper()}.EU.tenv3"
        wget.download(url, out=f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    if ((i+1)%2==0):
        EQ=EQ+1

1
GNSSVerify/EQ1.1/
Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv
100% [............................................................................] 300696 / 3006961
GNSSVerify/EQ1.2/
Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv
100% [............................................................................] 300696 / 3006962
GNSSVerify/EQ2.1/
Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv
100% [............................................................................] 608736 / 6087362
GNSSVerify/EQ2.2/
Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv
100% [............................................................................] 608736 / 6087363
GNSSVerify/EQ3.1/
Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv
100% [.......................

# Computing Coseismic Deformation from GNSS time-series

In [2]:
%%bash

t1_list=( # A year before event
    None
    2005.0192 # 2006-01-08
    2007.1202 # 2008-02-14
    2007.1202 # 2008-02-14
    2012.7781 # 2013-10-12
    2013.3918 # 2014-05-24
    2014.8767 # 2015-11-17
    2016.5479 # 2017-07-20
    2017.8137 # 2018-10-25
    2019.3333 # 2020-05-02
    2019.8279 # 2020-10-30
)

t2_list=( # A year after event
    None
    2007.0192 # 2006-01-08
    2009.1202 # 2008-02-14
    2009.1202 # 2008-02-14
    2014.7781 # 2013-10-12
    2015.3918 # 2014-05-24
    2016.8767 # 2015-11-17
    2018.5479 # 2017-07-20
    2019.8137 # 2018-10-25
    2021.3333 # 2020-05-02
    2021.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables

for i in $(seq 1 10); do
    t1=${t1_list[$i]}
    t2=${t2_list[$i]}

    for j in $(seq 1 2); do
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder

        cd $dirEXE # Go to executables folder
        files=("$dirIN"/*.EU.tenv3)

        # For every .tenv3 file in the folder
        for f in "${files[@]}"; do
            sta=$(basename "$f" .tenv3) # Station Code/ Identifier
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 11:North to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c="$11" 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            octave -q < input_cycleslip.m # Find the Cycle Slip
            cp serv.bayes $dirOUT/$sta.N.bayes.out # Save in output directory
            cp serv.p_tau $dirOUT/$sta.N.bayes.p_tau # Save in output directory
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 10:East to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c="$10" 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            octave -q < input_cycleslip.m # Find the Cycle Slip
            cp serv.bayes $dirOUT/$sta.E.bayes.out # Save in output directory
            cp serv.p_tau $dirOUT/$sta.E.bayes.p_tau # Save in output directory
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 12:Up to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c="$12" 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            octave -q < input_cycleslip.m # Find the Cycle Slip
            cp serv.bayes $dirOUT/$sta.U.bayes.out # Save in output directory
            cp serv.p_tau $dirOUT/$sta.U.bayes.p_tau # Save in output directory
        
            gawk "/ "$f"
        done
    done
done

error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory


/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify/EQ4.2/KAPS.EU.tenv3


error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory


/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify/EQ4.2/KISM.EU.tenv3


error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory


...

/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify/EQ10.2/YLDZ.EU.tenv3


error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory
error: load: unable to determine file format of 'serv.inp'
cp: cannot stat 'serv.bayes': No such file or directory
cp: cannot stat 'serv.p_tau': No such file or directory


/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify/EQ10.2/YSST.EU.tenv3
tau0 = 2019.939800000000
tau0 = 2019.939800000000
tau0 = 2019.835700000000


    posterior_cont_gen at line 177 column 9

    posterior_cont_gen at line 183 column 15



/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify/EQ10.2/ZKRO.EU.tenv3


## Read the displacement corresponding to each earthquake and write to a table

In [22]:
%%bash

events=(
    None
    2006.0192 # 2006-01-08
    2008.1202 # 2008-02-14
    2008.1202 # 2008-02-14
    2013.7781 # 2013-10-12
    2014.3918 # 2014-05-24
    2015.8767 # 2015-11-17
    2017.5479 # 2017-07-20
    2018.8137 # 2018-10-25
    2020.3333 # 2020-05-02
    2020.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE}

for i in $(seq 1 10); do # For each quake
    event=${events[$i]}
    echo $event | tee -a GNSSVerify.txt

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo EQ${i}.${j} | tee -a GNSSVerify.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*.EU.${g}.bayes.p_tau)
            echo ${g} | tee -a GNSSVerify.txt
    
            # For every .p_tau file
            for f in "${files[@]}"; do
                sta=$(basename "$f" .EU.${g}.bayes.p_tau) # Station Code/ Identifier
                { echo -n "$sta"; gawk -v val="${event}" 'sprintf("%.5f", $1) >= sprintf("%.5f", val) { print; exit }' "$f"; } | tee -a GNSSVerify.txt
            done 
        done
    done
done

2006.0192
EQ1.1
E
AKYR     2006.17110             NaN         0.00000         0.00000 
ANOP     2006.02050             NaN         0.00000         0.00000 
ATRS     2006.02050             NaN         0.00000         0.00000 
GVDS     2006.48600             NaN         0.00000         0.00000 
KERY     2006.02050             NaN         0.00000         0.00000 
KITH     2006.02050             NaN         0.00000         0.00000 
KOUN     2006.45860             NaN         0.00000         0.00000 
KRYO     2006.02050             NaN         0.00000         0.00000 
MEN1     2006.02050             NaN         0.00000         0.00000 
MET4     2006.02050             NaN         0.00000         0.00000 
NEA1     2006.02050             NaN         0.00000         0.00000 
PSAR     2006.44220             NaN         0.00000         0.00000 
RLSO     2006.57630             NaN         0.00000         0.00000 
SPR2     2006.55170             NaN         0.00000         0.00000 
TUC2     2006.02

## Read the date of the detected cycle slip

In [23]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE}

for i in $(seq 1 10); do # For each quake

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo EQ${i}.${j} | tee -a CycleSlip.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*.EU.${g}.bayes.out)
            echo ${g} | tee -a CycleSlip.txt
    
            # For every .p_tau file
            for f in "${files[@]}"; do
                sta=$(basename "$f" .EU.${g}.bayes.out) # Station Code/ Identifier
                { echo -n "$sta"; gawk 'NR == 2 {print}' "$f"; } | tee -a CycleSlip.txt
            done 
        done
    done
done

EQ1.1
E
AKYREpoca tau0 della discontinuit�   =  2005.1691
ANOPEpoca tau0 della discontinuit�   =  2005.1663
ATRSEpoca tau0 della discontinuit�   =  2005.4018
GVDSEpoca tau0 della discontinuit�   =  2006.5188
KERYEpoca tau0 della discontinuit�   =  2005.0486
KITHEpoca tau0 della discontinuit�   =  2005.0486
KOUNEpoca tau0 della discontinuit�   =  2005.0486
KRYOEpoca tau0 della discontinuit�   =  2005.3881
MEN1Epoca tau0 della discontinuit�   =  2005.0486
MET4Epoca tau0 della discontinuit�   =  2005.0486
NEA1Epoca tau0 della discontinuit�   =  2005.0486
PSAREpoca tau0 della discontinuit�   =  2005.0486
RLSOEpoca tau0 della discontinuit�   =  2006.6037
SPR2Epoca tau0 della discontinuit�   =  2006.5791
TUC2Epoca tau0 della discontinuit�   =  2005.0486
VASSEpoca tau0 della discontinuit�   =  2005.0486
XRSOEpoca tau0 della discontinuit�   =  2005.0486
N
AKYREpoca tau0 della discontinuit�   =  2005.1691
ANOPEpoca tau0 della discontinuit�   =  2005.1663
ATRSEpoca tau0 della discontinuit�   =  